<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/05_synthetic_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Synthetic Data Validation
Generate a synthetic MMM dataset with known ground-truth ROI parameters, fit the model, and prepare outputs for attribution validation in notebook 06.

## Generate Synthetic Dataset

In [ ]:
import numpy as np
import pandas as pd

def generate_synthetic_mmm_data(n_weeks=104, start_date="2022-01-03", seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start_date, periods=n_weeks, freq="W-MON")
    meta_spend   = rng.lognormal(mean=9.5, sigma=0.4, size=n_weeks)
    google_spend = rng.lognormal(mean=9.8, sigma=0.35, size=n_weeks)
    meta_imp     = (meta_spend   * rng.uniform(80, 120, n_weeks)).astype(int)
    google_imp   = (google_spend * rng.uniform(60,  90, n_weeks)).astype(int)
    offline      = rng.choice([0, 0.05, 0.1, 0.15], n_weeks, p=[0.6, 0.2, 0.1, 0.1])

    def adstock(x, d=0.5):
        out = np.zeros_like(x); out[0] = x[0]
        for t in range(1, len(x)): out[t] = x[t] + d * out[t-1]
        return out

    t = np.arange(n_weeks)
    season  = 1 + 0.15*np.sin(2*np.pi*t/52) + 0.08*np.cos(4*np.pi*t/52)
    revenue = (500_000 + 0.8*adstock(meta_imp,.4) + 1.2*adstock(google_imp,.3) + 200_000*offline)*season + rng.normal(0,20_000,n_weeks)

    return pd.DataFrame({
        "date":             dates.strftime("%Y/%m/%d"),
        "Meta_spend":       meta_spend.round(2),
        "Google_spend":     google_spend.round(2),
        "Meta_impression":  meta_imp,
        "Google_impression":google_imp,
        "Offline Discount": offline,
        "Revenue":          np.clip(revenue, 0, None).round(2),
    })

df = generate_synthetic_mmm_data(n_weeks=104, seed=42)
df.to_csv('data/synthetic_mmm_data.csv', index=False)
print(df.shape)
df.head()

## Data Quality Check

In [ ]:
def detect_columns(data):
    media = [c for c in data.columns if any(k in c.lower() for k in ['spend','cost','budget']) and data[c].sum() > 0]
    impressions = [c for c in data.columns if 'impression' in c.lower() or 'impresseion' in c.lower()]
    output = [c for c in data.columns if 'revenue' in c.lower()]
    return media, impressions, output

def check_data_quality(data):
    print("=== Missing Values ===")
    missing = data.isnull().sum()
    print(missing[missing > 0] if missing.any() else "No missing values.")
    print("\n=== Date Range ===")
    if 'date' in data.columns:
        print(f"  {data['date'].min()} → {data['date'].max()}  ({len(data)} rows)")

media, impressions, output = detect_columns(df)
check_data_quality(df)
print('\nMedia:', media)
print('Impressions:', impressions)
print('KPI:', output)

## Fit Model on Synthetic Data

In [ ]:
import numpy as np, tensorflow_probability as tfp
from meridian import constants
from meridian.data import load
from meridian.model import model, spec, prior_distribution

def build_channel_mappings(media, impressions):
    def strip(col, suffixes):
        for s in suffixes:
            if col.lower().endswith(s): return col[:-len(s)]
        return col
    cost_mapping        = {c: strip(c, ['spend','cost','budget']) for c in media}
    impressions_mapping = {c: strip(c, ['impression','impresseion']) for c in impressions}
    return cost_mapping, impressions_mapping

cost_mapping, impressions_mapping = build_channel_mappings(media, impressions)

controls = [c for c in ['Offline Discount'] if c in df.columns]
coord_to_columns = load.CoordToColumns(
    time='date', kpi=output[0],
    controls=controls if controls else None,
    media=impressions, media_spend=media,
)
loader = load.DataFrameDataLoader(
    df=df, kpi_type='revenue',
    coord_to_columns=coord_to_columns,
    media_spend_to_channel=cost_mapping,
    media_to_channel=impressions_mapping,
)
data_meridian = loader.load()

def estimate_lognormal_dist(mean, std):
    mu_log  = np.log(mean) - 0.5 * np.log((std/mean)**2 + 1)
    std_log = np.sqrt(np.log((std/mean)**2 + 1))
    return mu_log, std_log

roi_mu_log, roi_sigma_log = estimate_lognormal_dist(2.0, 1.5)
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=np.float32(roi_mu_log), scale=np.float32(roi_sigma_log), name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, max_lag=7)
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)
mmm.sample_prior(50)
mmm.sample_posterior(n_chains=3, n_adapt=500, n_burnin=500, n_keep=1000)

## Next Step
Run **`06_attribution_validation.ipynb`** to compare estimated ROI against ground-truth values.